In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
df = pd.read_csv("customer_support_tickets_200k (1).csv")

In [3]:
print(df.shape)

df.head()

(200000, 30)


,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,...,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,Urgent,Open,Email,...,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Urgent,Closed,Email,...,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business
2,3,William Anderson,william.anderson651@outlook.com,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Chat,...,2022-12-05,Yes,Yes,Windows,Safari,Bank Transfer,French,Morning,4,Corporate
3,4,David Miller,david.miller672@icloud.com,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my reque...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Social Media,...,2024-04-04,Yes,No,Windows,Chrome,Credit Card,Spanish,Afternoon,7,Corporate
4,5,Robert Gonzalez,robert.gonzalez391@hotmail.com,Web Portal,Feature Request,The system is not syncing data across devices ...,We have reset the account credentials and advi...,High,Pending Customer,Email,...,2024-08-24,Yes,No,Linux,NaN,Debit Card,Spanish,Evening,3,Corporate


In [4]:
print(df.columns)

Index(['ticket_id', 'customer_name', 'customer_email', 'product', 'category',
       'issue_description', 'resolution_notes', 'priority', 'status',
       'channel', 'region', 'customer_age', 'customer_gender',
       'subscription_type', 'customer_tenure_months', 'previous_tickets',
       'customer_satisfaction_score', 'first_response_time_hours',
       'resolution_time_hours', 'ticket_created_date', 'ticket_resolved_date',
       'escalated', 'sla_breached', 'operating_system', 'browser',
       'payment_method', 'language', 'preferred_contact_time',
       'issue_complexity_score', 'customer_segment'],
      dtype='object')


In [5]:
df = df[['issue_description','category','priority']]

In [6]:
print(df.isnull().sum())

issue_description    0
category             0
priority             0
dtype: int64


In [7]:
df = df.dropna()

In [8]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'http\S+', '', text)

    text = re.sub(r'[^a-zA-Z ]', '', text)

    text = text.split()

    return " ".join(text)

In [9]:
df['cleaned_text'] = df['issue_description'].apply(clean_text)

In [10]:
df[['issue_description','cleaned_text']].head()

,issue_description,cleaned_text
0,The payment was deducted from my bank account ...,the payment was deducted from my bank account ...
1,I found a bug in the latest update affecting r...,i found a bug in the latest update affecting r...
2,The application crashes whenever I try to uplo...,the application crashes whenever i try to uplo...
3,My subscription was cancelled without my reque...,my subscription was cancelled without my reque...
4,The system is not syncing data across devices ...,the system is not syncing data across devices ...


In [11]:
X = df['cleaned_text']

y = df['category']

In [12]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X_tfidf = vectorizer.fit_transform(X)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
model = LogisticRegression(
    max_iter=1000
)

model.fit(
    X_train,
    y_train
)

LogisticRegression(max_iter=1000)

In [15]:
y_pred = model.predict(X_test)

In [17]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy =", accuracy)

Accuracy = 0.097475


In [20]:
print(
    classification_report(y_test,y_pred,zero_division=0)
)

                           precision    recall  f1-score   support

       Account Suspension       0.10      0.10      0.10      3973
               Bug Report       0.00      0.00      0.00      4009
          Data Sync Issue       0.00      0.00      0.00      4013
          Feature Request       0.09      0.19      0.13      3941
              Login Issue       0.00      0.00      0.00      4022
          Payment Problem       0.09      0.09      0.09      4006
        Performance Issue       0.09      0.10      0.10      3967
           Refund Request       0.10      0.19      0.13      3983
         Security Concern       0.10      0.20      0.13      3949
Subscription Cancellation       0.11      0.11      0.11      4137

                 accuracy                           0.10     40000
                macro avg       0.07      0.10      0.08     40000
             weighted avg       0.07      0.10      0.08     40000



In [21]:
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

[[384   0   0 779   0 415 414 789 787 405]
 [373   0   0 835   0 419 421 822 781 358]
 [393   0   0 872   0 410 393 746 780 419]
 [385   0   0 765   0 376 405 811 793 406]
 [438   0   0 796   0 385 389 787 795 432]
 [435   0   0 810   0 373 411 778 809 390]
 [382   0   0 803   0 423 383 766 822 388]
 [413   0   0 806   0 391 391 776 812 394]
 [410   0   0 758   0 376 422 806 781 396]
 [429   0   0 841   0 382 414 782 852 437]]


In [22]:
X_priority = df['cleaned_text']

y_priority = df['priority']

In [23]:
X_priority_tfidf = vectorizer.fit_transform(
    X_priority
)

In [24]:
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X_priority_tfidf,
    y_priority,
    test_size=0.2,
    random_state=42
)

In [25]:
priority_model = LogisticRegression(
    max_iter=1000
)

priority_model.fit(
    Xp_train,
    yp_train
)

LogisticRegression(max_iter=1000)

In [26]:
priority_pred = priority_model.predict(
    Xp_test
)

In [27]:
print(
    accuracy_score(
        yp_test,
        priority_pred
    )
)

0.24955


In [29]:
print(
    classification_report(
        yp_test,
        priority_pred,zero_division=0
    )
)

              precision    recall  f1-score   support

        High       0.26      0.30      0.28     10152
         Low       0.25      0.20      0.22      9885
      Medium       0.00      0.00      0.00     10062
      Urgent       0.25      0.50      0.33      9901

    accuracy                           0.25     40000
   macro avg       0.19      0.25      0.21     40000
weighted avg       0.19      0.25      0.21     40000



In [30]:
new_ticket = [
    "Payment failed during checkout process"
]

In [31]:
clean_ticket = clean_text(
    new_ticket[0]
)

In [32]:
vector_ticket = vectorizer.transform(
    [clean_ticket]
)